# Day 4 · Sentence Embeddings — From Words to Meaning

**Goal:** Move beyond bag-of-words. Use a pre-trained transformer to turn sentences into vectors, then measure how "close" their meanings are using cosine similarity.

In [ ]:
# ── 1. Install & Import ──────────────────────────────────────────────
!pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

print("ready ✅")

In [ ]:
# ── 2. Load model & define sentences ─────────────────────────────────
model = SentenceTransformer('all-MiniLM-L6-v2')   # small, fast, good quality

sentences = [
    "The cat sat on the mat.",                          # S0 – animals / simple
    "A kitten was resting on the rug.",                 # S1 – very similar to S0
    "Deep learning models require large datasets.",     # S2 – tech / ML
    "Neural networks need lots of training data.",      # S3 – very similar to S2
    "The stock market rallied after the Fed announcement."  # S4 – finance (outlier)
]

for i, s in enumerate(sentences):
    print(f"S{i}: {s}")

In [ ]:
# ── 3. Generate embeddings ───────────────────────────────────────────
embeddings = model.encode(sentences)

print(f"Shape: {embeddings.shape}")
print(f"  → {len(sentences)} sentences, each mapped to a {embeddings.shape[1]}-dim vector")
print(f"\nFirst 10 values of S0's embedding:")
print(np.round(embeddings[0][:10], 4))

In [ ]:
# ── 4. Cosine-similarity matrix ─────────────────────────────────────
sim_matrix = cosine_similarity(embeddings)

# pretty-print
labels = [f"S{i}" for i in range(len(sentences))]
header = "     " + "  ".join(f"{l:>6}" for l in labels)
print(header)
for i, row in enumerate(sim_matrix):
    vals = "  ".join(f"{v:6.3f}" for v in row)
    print(f"{labels[i]:>4}  {vals}")

In [ ]:
# ── 5. Rank all pairs by similarity ─────────────────────────────────
pairs = []
for i in range(len(sentences)):
    for j in range(i + 1, len(sentences)):
        pairs.append((i, j, sim_matrix[i][j]))

pairs.sort(key=lambda x: x[2], reverse=True)

print("All sentence pairs ranked by cosine similarity:\n")
for i, j, score in pairs:
    print(f"  S{i} ↔ S{j}  →  {score:.4f}")
    print(f"      \"{sentences[i]}\"")
    print(f"      \"{sentences[j]}\"\n")

## Reflection

**Highest-scoring pairs:**
- **S2 ↔ S3** (deep learning ↔ neural networks) — these share almost identical *meaning* even though they use different words. Bag-of-words would score them low (few overlapping tokens), but embeddings see right through the paraphrasing.
- **S0 ↔ S1** (cat on mat ↔ kitten on rug) — again, synonyms (cat/kitten, mat/rug) are captured.

**Lowest-scoring pairs:**
- Any pair involving **S4** (stock market) scores much lower because its topic is entirely different.

**What surprised me:**
The model treats "cat" and "kitten" as near-synonyms and "mat" and "rug" similarly, producing a high score despite zero word overlap beyond "the." This is exactly the leap from bag-of-words → embeddings: **meaning over surface form**. It also surprised me how clearly the finance sentence (S4) separates from everything else — the model creates a real "topic boundary" in vector space.